<a href="https://colab.research.google.com/github/I7411/TH_BIGDATA/blob/main/HuynhThanhMinhTam_2001230789_BT1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bước 0. Khởi tạo Colab mới, SparkSession và nạp dữ liệu nguồn từ Drive

In [5]:
!pip -q install pyspark psutil

import os
import shutil
import statistics
import time
from pathlib import Path

import psutil
from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from pyspark.sql import SparkSession, Window, functions as F, types as T

# 1. Khởi tạo SparkSession (cố định cấu hình local[2] như Bài 1)
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("Lab02_Bai2_SummitRoute_ETL")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("CPU cores:", os.cpu_count())
print("RAM GiB:", round(psutil.virtual_memory().total / (2**30), 2))
print("Master:", spark.sparkContext.master)

# 2. Tạo cấu trúc thư mục Data Lake (Raw, Cleaned, Curated)
ROOT = Path("/content/lab02_bai2/data_lake")
RAW = ROOT / "raw"
CLEANED = ROOT / "cleaned"
CURATED = ROOT / "curated"

for folder in (RAW, CLEANED, CURATED):
    folder.mkdir(parents=True, exist_ok=True)

FILENAME = "Lab02_Bai2_US_Logistics_Shipments.csv"
DATA_PATH = RAW / FILENAME

# 3. Nạp tệp CSV vào vùng Raw (hỗ trợ cả tệp .csv trong Drive hoặc Google Sheets)
drive.mount("/content/drive")
source_csv = Path("/content/drive/MyDrive/BigData_Lab02") / FILENAME

if source_csv.is_file():
    shutil.copy2(source_csv, DATA_PATH)
else:
    auth.authenticate_user()
    drive_service = build("drive", "v3")
    res = drive_service.files().list(
        q="name contains 'Lab02_Bai2_US_Logistics_Shipments' and trashed = false",
        fields="files(id, name, mimeType)",
        pageSize=5,
    ).execute()
    matched = res.get("files", [])
    assert matched, "Không tìm thấy tệp Lab02_Bai2_US_Logistics_Shipments trên Google Drive!"

    # Ưu tiên chọn tệp CSV gốc (text/csv) để tránh giới hạn kích thước xuất của Google Sheets
    csv_files = [f for f in matched if f["mimeType"] != "application/vnd.google-apps.spreadsheet"]
    if csv_files:
        target = csv_files[0]
    else:
        target = matched[0]

    if target["mimeType"] == "application/vnd.google-apps.spreadsheet":
        request = drive_service.files().export_media(
            fileId=target["id"], mimeType="text/csv"
        )
    else:
        request = drive_service.files().get_media(fileId=target["id"])

    with open(DATA_PATH, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

assert DATA_PATH.is_file(), f"Thiếu tệp nguồn: {DATA_PATH}"
print(f"Đã nạp vùng Raw: {DATA_PATH} ({round(DATA_PATH.stat().st_size / (2**20), 2)} MiB)")

Spark version: 4.0.5
CPU cores: 2
RAM GiB: 12.67
Master: local[2]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã nạp vùng Raw: /content/lab02_bai2/data_lake/raw/Lab02_Bai2_US_Logistics_Shipments.csv (63.36 MiB)


Câu 1. Khảo sát nguồn dữ liệu với schema tường minh

In [6]:
# Khai báo schema tường minh đúng thứ tự 21 cột của tập dữ liệu SummitRoute Logistics
RAW_SCHEMA = T.StructType([
    T.StructField("shipment_id", T.LongType()),
    T.StructField("ship_date", T.StringType()),
    T.StructField("customer_id", T.LongType()),
    T.StructField("origin_city", T.StringType()),
    T.StructField("origin_state", T.StringType()),
    T.StructField("destination_city", T.StringType()),
    T.StructField("destination_state", T.StringType()),
    T.StructField("service_level", T.StringType()),
    T.StructField("package_type", T.StringType()),
    T.StructField("weight_kg", T.DoubleType()),
    T.StructField("distance_km", T.DoubleType()),
    T.StructField("base_fee_usd", T.DoubleType()),
    T.StructField("fuel_fee_usd", T.DoubleType()),
    T.StructField("insurance_fee_usd", T.DoubleType()),
    T.StructField("total_fee_usd", T.DoubleType()),
    T.StructField("promised_days", T.IntegerType()),
    T.StructField("actual_days", T.IntegerType()),
    T.StructField("delivery_status", T.StringType()),
    T.StructField("scan_events_json", T.StringType()),
    T.StructField("label_tint", T.StringType()),
    T.StructField("scanner_batch", T.StringType()),
])

raw = (
    spark.read.schema(RAW_SCHEMA)
    .option("header", "true")
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(str(DATA_PATH))
)

# Tính các chỉ số khảo sát bằng PySpark
total_rows = raw.count()
total_cols = len(raw.columns)
distinct_shipments = raw.select("shipment_id").distinct().count()

stats_row = (
    raw.withColumn("_dt", F.to_date("ship_date", "yyyy-MM-dd"))
    .agg(
        F.min("_dt").alias("min_date"),
        F.max("_dt").alias("max_date"),
        F.countDistinct("origin_state").alias("origin_states"),
        F.countDistinct("destination_state").alias("dest_states"),
    )
    .first()
)

print("=== BÁO CÁO KHẢO SÁT NGUỒN DỮ LIỆU (CÂU 1) ===")
print(f"- Tổng số dòng: {total_rows:,}")
print(f"- Tổng số cột: {total_cols}")
print(f"- Số shipment_id phân biệt: {distinct_shipments:,}")
print(f"- Khoảng thời gian ghi nhận: {stats_row['min_date']} đến {stats_row['max_date']}")
print(f"- Số bang xuất phát (origin_state): {stats_row['origin_states']}")
print(f"- Số bang đích (destination_state): {stats_row['dest_states']}")
raw.printSchema()
raw.select("shipment_id", "ship_date", "origin_state", "destination_state", "total_fee_usd", "scan_events_json").show(3, truncate=85)

=== BÁO CÁO KHẢO SÁT NGUỒN DỮ LIỆU (CÂU 1) ===
- Tổng số dòng: 300,000
- Tổng số cột: 21
- Số shipment_id phân biệt: 300,000
- Khoảng thời gian ghi nhận: 2025-01-01 đến 2025-12-31
- Số bang xuất phát (origin_state): 6
- Số bang đích (destination_state): 8
root
 |-- shipment_id: long (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_type: string (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- base_fee_usd: double (nullable = true)
 |-- fuel_fee_usd: double (nullable = true)
 |-- insurance_fee_usd: double (nullable = true)
 |-- total_fee_usd: double (nullable = true)
 |-- promised_days: integer (nullable = true)
 |-- actual_days: integer (null

Câu 2. Thiết kế tầng Raw và tạo bộ JSON Lines có cấu trúc

In [7]:
# Định nghĩa schema cho cấu trúc lồng nhau của scan_events_json
SCAN_SCHEMA = T.StructType([
    T.StructField("scan", T.StructType([
        T.StructField("hub", T.StringType()),
        T.StructField("mode", T.StringType()),
    ])),
    T.StructField("stages", T.ArrayType(T.StringType())),
])

parsed = raw.withColumn(
    "scan_events", F.from_json(F.col("scan_events_json"), SCAN_SCHEMA)
)

# Xuất bộ JSON Lines lưu scan_events dưới dạng object có cấu trúc
JSON_DIR = ROOT / "json_events"
export_json = parsed.select(
    "shipment_id",
    "ship_date",
    "service_level",
    "destination_state",
    "scan_events",
)

(
    export_json.coalesce(4)
    .write.mode("overwrite")
    .json(str(JSON_DIR))
)

# Đọc lại từ JSON Lines và kiểm tra số dòng + schema
json_df = spark.read.json(str(JSON_DIR))
print("Số dòng đọc lại từ JSON Lines:", f"{json_df.count():,}")
json_df.printSchema()
json_df.select("shipment_id", "scan_events.scan.hub", "scan_events.scan.mode", "scan_events.stages").show(5, truncate=False)

Số dòng đọc lại từ JSON Lines: 300,000
root
 |-- destination_state: string (nullable = true)
 |-- scan_events: struct (nullable = true)
 |    |-- scan: struct (nullable = true)
 |    |    |-- hub: string (nullable = true)
 |    |    |-- mode: string (nullable = true)
 |    |-- stages: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- service_level: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- shipment_id: long (nullable = true)

+-----------+---+------+-------------------------------+
|shipment_id|hub|mode  |stages                         |
+-----------+---+------+-------------------------------+
|810000001  |DAL|Manual|[Pickup, Hub, Delay, Delivered]|
|810000002  |CHI|Auto  |[Pickup, Hub, Delivered]       |
|810000003  |ATL|Auto  |[Pickup, Hub, Delivered]       |
|810000004  |SFO|Auto  |[Pickup, Hub, Delivered]       |
|810000005  |PHL|Auto  |[Pickup, Hub, Delivered]       |
+-----------+---+------+-----------------------------

Câu 3. Trích xuất dữ liệu bán cấu trúc và phân tích nguy cơ nhân bản chi phí

In [8]:
# Trích xuất scan.hub, scan.mode và tách mảng stages[] bằng explode_outer
exploded_stages = parsed.select(
    "shipment_id",
    "service_level",
    "total_fee_usd",
    F.col("scan_events.scan.hub").alias("scan_hub"),
    F.col("scan_events.scan.mode").alias("scan_mode"),
    F.explode_outer(F.col("scan_events.stages")).alias("stage"),
)

rows_before = parsed.count()
rows_after = exploded_stages.count()

print(f"Số dòng trước khi tách mảng stages[]: {rows_before:,}")
print(f"Số dòng sau khi tách mảng stages[]:   {rows_after:,} (tăng {rows_after - rows_before:,} dòng)")

print("\nPhân bố theo từng chặng quét (stage):")
exploded_stages.groupBy("stage").count().orderBy(F.desc("count")).show()

print("Phân bố theo trạm quét và chế độ quét (scan.hub, scan.mode):")
exploded_stages.select("shipment_id", "scan_hub", "scan_mode").distinct().groupBy("scan_hub", "scan_mode").count().orderBy("scan_hub", "scan_mode").show()

Số dòng trước khi tách mảng stages[]: 300,000
Số dòng sau khi tách mảng stages[]:   925,932 (tăng 625,932 dòng)

Phân bố theo từng chặng quét (stage):
+---------+------+
|    stage| count|
+---------+------+
|      Hub|300000|
|Delivered|300000|
|   Pickup|300000|
|    Delay| 25932|
+---------+------+

Phân bố theo trạm quét và chế độ quét (scan.hub, scan.mode):
+--------+---------+-----+
|scan_hub|scan_mode|count|
+--------+---------+-----+
|     ATL|     Auto|42857|
|     ATL|   Manual|14286|
|     CHI|     Auto|42857|
|     DAL|     Auto|42857|
|     DAL|   Manual|14286|
|     DEN|     Auto|42857|
|     PHL|     Auto|42857|
|     PHL|   Manual|14286|
|     SFO|     Auto|42857|
+--------+---------+-----+



Mỗi vận đơn (shipment_id) chỉ phát sinh một khoản phí vận chuyển total_fee_usd duy nhất, nhưng mảng stages[] của vận đơn đó lại chứa nhiều chặng quét liên tiếp (ví dụ ["Pickup", "Hub", "Delivered"] gồm 3 chặng, hoặc 4 chặng nếu có "Delay").   

Khi dùng explode_outer trên cột stages, một vận đơn có $k$ chặng sẽ bị nhân bản thành $k$ dòng riêng biệt và giá trị total_fee_usd được sao chép nguyên vẹn sang cả $k$ dòng đó. Nếu tính sum(total_fee_usd) trực tiếp trên bảng đã tách mảng mà không khử trùng theo shipment_id hoặc tính trước khi explode, tổng chi phí vận chuyển sẽ bị cộng lặp $k$ lần (fan-out duplication), làm sai lệch nghiêm trọng báo cáo tài chính.   

Câu 4. Data Quality Gate (Kiểm tra 6 tiêu chí chất lượng dữ liệu)

In [9]:
# Cửa sổ kiểm tra trùng lặp shipment_id
w_id = Window.partitionBy("shipment_id")

checked = (
    parsed
    .withColumn("_id_count", F.count("*").over(w_id))
    .withColumn("ship_dt", F.to_date("ship_date", "yyyy-MM-dd"))
    .withColumn(
        "expected_fee",
        F.round(
            F.col("base_fee_usd") + F.col("fuel_fee_usd") + F.col("insurance_fee_usd"),
            2,
        ),
    )
    # 6 tiêu chí kiểm tra lỗi riêng biệt
    .withColumn("err_dup_id", F.col("shipment_id").isNull() | (F.col("_id_count") > 1))
    .withColumn("err_bad_date", F.col("ship_dt").isNull())
    .withColumn(
        "err_bad_measure",
        F.col("weight_kg").isNull()
        | (F.col("weight_kg") <= 0)
        | F.col("distance_km").isNull()
        | (F.col("distance_km") <= 0),
    )
    .withColumn(
        "err_bad_json",
        F.col("scan_events").isNull()
        | F.col("scan_events.scan.hub").isNull()
        | F.col("scan_events.scan.mode").isNull()
        | F.col("scan_events.stages").isNull()
        | (F.size(F.col("scan_events.stages")) == 0),
    )
    .withColumn(
        "err_bad_fee",
        F.col("total_fee_usd").isNull()
        | F.col("expected_fee").isNull()
        | (F.abs(F.col("total_fee_usd") - F.col("expected_fee")) > 0.011),
    )
    .withColumn(
        "err_bad_status",
        F.col("promised_days").isNull()
        | F.col("actual_days").isNull()
        | F.col("delivery_status").isNull()
        | ~(
            ((F.col("actual_days") <= F.col("promised_days")) & (F.col("delivery_status") == "OnTime"))
            | ((F.col("actual_days") > F.col("promised_days")) & (F.col("delivery_status") == "Late"))
        ),
    )
)

checked = checked.withColumn(
    "valid_record",
    ~(
        F.col("err_dup_id")
        | F.col("err_bad_date")
        | F.col("err_bad_measure")
        | F.col("err_bad_json")
        | F.col("err_bad_fee")
        | F.col("err_bad_status")
    ),
)

# Tổng hợp bảng số dòng lỗi theo từng tiêu chí trong 1 lượt quét PySpark
dq_stats = checked.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.col("err_dup_id").cast("int")).alias("c1_dup_id"),
    F.sum(F.col("err_bad_date").cast("int")).alias("c2_bad_date"),
    F.sum(F.col("err_bad_measure").cast("int")).alias("c3_bad_measure"),
    F.sum(F.col("err_bad_json").cast("int")).alias("c4_bad_json"),
    F.sum(F.col("err_bad_fee").cast("int")).alias("c5_bad_fee"),
    F.sum(F.col("err_bad_status").cast("int")).alias("c6_bad_status"),
    F.sum((~F.col("valid_record")).cast("int")).alias("total_invalid"),
    F.sum(F.col("valid_record").cast("int")).alias("total_valid"),
).first()

print("=== BẢNG KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY GATE) ===")
dq_rows = [
    ("1. shipment_id trùng hoặc rỗng", dq_stats["c1_dup_id"]),
    ("2. Ngày gửi (ship_date) không hợp lệ", dq_stats["c2_bad_date"]),
    ("3. weight_kg hoặc distance_km <= 0 / null", dq_stats["c3_bad_measure"]),
    ("4. Dữ liệu JSON scan_events lỗi cấu trúc", dq_stats["c4_bad_json"]),
    ("5. Tổng phí lệch tổng 3 thành phần (> 0.011 USD)", dq_stats["c5_bad_fee"]),
    ("6. delivery_status mâu thuẫn promised_days/actual_days", dq_stats["c6_bad_status"]),
    ("Tổng số dòng không đạt (bị loại)", dq_stats["total_invalid"]),
    ("Tổng số dòng đạt yêu cầu (valid_record = True)", dq_stats["total_valid"]),
]
for label, cnt in dq_rows:
    print(f"{label:<55}: {cnt:,}")

# Lưu vùng cách ly (quarantine) để truy vết nếu có bản ghi lỗi
QUARANTINE_DIR = ROOT / "quarantine"
checked.filter(~F.col("valid_record")).write.mode("overwrite").parquet(str(QUARANTINE_DIR))

=== BẢNG KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (DATA QUALITY GATE) ===
1. shipment_id trùng hoặc rỗng                         : 0
2. Ngày gửi (ship_date) không hợp lệ                   : 0
3. weight_kg hoặc distance_km <= 0 / null              : 0
4. Dữ liệu JSON scan_events lỗi cấu trúc               : 0
5. Tổng phí lệch tổng 3 thành phần (> 0.011 USD)       : 0
6. delivery_status mâu thuẫn promised_days/actual_days : 0
Tổng số dòng không đạt (bị loại)                       : 0
Tổng số dòng đạt yêu cầu (valid_record = True)         : 300,000


Câu 5. Tạo vùng Cleaned (Silver) và ghi Parquet

In [10]:
silver = (
    checked.filter(F.col("valid_record"))
    # Giữ đầy đủ cột nghiệp vụ & cấu trúc scan_events; loại bỏ cột tạm và 2 cột gây nhiễu
    .drop(
        "ship_date",
        "scan_events_json",
        "label_tint",
        "scanner_batch",
        "_id_count",
        "expected_fee",
        "err_dup_id",
        "err_bad_date",
        "err_bad_measure",
        "err_bad_json",
        "err_bad_fee",
        "err_bad_status",
        "valid_record",
    )
    .withColumnRenamed("ship_dt", "ship_date")
    .dropDuplicates(["shipment_id"])
    .withColumn("year", F.year("ship_date"))
    .withColumn("month", F.month("ship_date"))
)

SILVER_DIR = CLEANED / "shipments_parquet"
silver.write.mode("overwrite").parquet(str(SILVER_DIR))

silver_read = spark.read.parquet(str(SILVER_DIR))
print("Số bản ghi Cleaned sau khi ghi/đọc Parquet:", f"{silver_read.count():,}")
silver_read.printSchema()

Số bản ghi Cleaned sau khi ghi/đọc Parquet: 300,000
root
 |-- shipment_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- origin_city: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_type: string (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- base_fee_usd: double (nullable = true)
 |-- fuel_fee_usd: double (nullable = true)
 |-- insurance_fee_usd: double (nullable = true)
 |-- total_fee_usd: double (nullable = true)
 |-- promised_days: integer (nullable = true)
 |-- actual_days: integer (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- scan_events: struct (nullable = true)
 |    |-- scan: struct (nullable = true)
 |    |    |-- hub: string (nullable = true)
 |    |    |-- mode: string (nullable = true)
 |    |-- 

Câu 6. Tạo vùng Curated (Gold) và đối chiếu tổng phí toàn tập

In [11]:
silver_clean = spark.read.parquet(str(SILVER_DIR))

gold = (
    silver_clean.groupBy("year", "month", "service_level", "destination_state")
    .agg(
        F.count("*").alias("shipments"),
        F.round(F.sum("total_fee_usd"), 2).alias("total_fee_usd"),
        F.round(F.avg("total_fee_usd"), 2).alias("avg_fee_usd"),
        F.sum(F.when(F.col("delivery_status") == "Late", 1).otherwise(0)).alias("late_shipments"),
        F.round(
            F.avg(F.when(F.col("delivery_status") == "Late", 1.0).otherwise(0.0)) * 100,
            2,
        ).alias("late_rate_pct"),
    )
)

GOLD_DIR = CURATED / "monthly_service_dest_kpis"
(
    gold.write.mode("overwrite")
    .partitionBy("year", "month")
    .parquet(str(GOLD_DIR))
)

gold_read = spark.read.parquet(str(GOLD_DIR))
print("Số nhóm bản ghi trong Curated:", f"{gold_read.count():,}")
gold_read.orderBy("year", "month", "service_level", "destination_state").show(10)

# Đối chiếu tổng phí toàn tập giữa Cleaned (nguồn đủ điều kiện) và Curated
cleaned_total_fee = silver_clean.agg(F.sum("total_fee_usd")).first()[0]
curated_total_fee = gold_read.agg(F.sum("total_fee_usd")).first()[0]

print(f"Tổng phí USD tại Cleaned: {cleaned_total_fee:,.2f}")
print(f"Tổng phí USD tại Curated: {curated_total_fee:,.2f}")
print(f"Chênh lệch tuyệt đối:     {abs(cleaned_total_fee - curated_total_fee):.4f} USD")
assert abs(cleaned_total_fee - curated_total_fee) < 1.0, "Lệch tổng phí giữa Cleaned và Curated!"

Số nhóm bản ghi trong Curated: 288
+-------------+-----------------+---------+-------------+-----------+--------------+-------------+----+-----+
|service_level|destination_state|shipments|total_fee_usd|avg_fee_usd|late_shipments|late_rate_pct|year|month|
+-------------+-----------------+---------+-------------+-----------+--------------+-------------+----+-----+
|      Economy|               CA|     1158|     34672.88|      29.94|            98|         8.46|2025|    1|
|      Economy|               MN|      772|     18378.35|      23.81|            70|         9.07|2025|    1|
|      Economy|               NC|     1159|     28634.71|      24.71|            98|         8.46|2025|    1|
|      Economy|               NY|     1157|     31297.33|      27.05|           101|         8.73|2025|    1|
|      Economy|               OH|      772|     18422.92|      23.86|            68|         8.81|2025|    1|
|      Economy|               TN|     1159|     25693.97|      22.17|           100| 

Câu 7. Lưu Cleaned dạng Partitioned Parquet và kiểm tra Partition Pruning

In [12]:
PART_DIR = CLEANED / "shipments_partitioned"
(
    silver_clean.repartition(4, "year", "month")
    .write.mode("overwrite")
    .partitionBy("year", "month")
    .parquet(str(PART_DIR))
)

partition_folders = sorted(PART_DIR.glob("year=*/month=*"))
part_files = [p for p in PART_DIR.rglob("*.parquet") if p.is_file()]

print(f"Tổng số thư mục phân vùng (year/month): {len(partition_folders)}")
print(f"Tổng số tệp dữ liệu (.parquet part-files): {len(part_files)}")
print("Danh sách các thư mục phân vùng:")
for folder in partition_folders:
    n_files = len(list(folder.glob("*.parquet")))
    print(f"  - {folder.relative_to(PART_DIR)} ({n_files} part-files)")

# Lọc riêng tháng 7/2025 và kiểm tra Physical Plan
partitioned_df = spark.read.parquet(str(PART_DIR))
july_query = (
    partitioned_df.filter((F.col("year") == 2025) & (F.col("month") == 7))
    .groupBy("service_level")
    .agg(
        F.count("*").alias("july_shipments"),
        F.round(F.sum("total_fee_usd"), 2).alias("july_fee_usd"),
    )
)

july_query.show()
july_query.explain(mode="formatted")

Tổng số thư mục phân vùng (year/month): 12
Tổng số tệp dữ liệu (.parquet part-files): 12
Danh sách các thư mục phân vùng:
  - year=2025/month=1 (1 part-files)
  - year=2025/month=10 (1 part-files)
  - year=2025/month=11 (1 part-files)
  - year=2025/month=12 (1 part-files)
  - year=2025/month=2 (1 part-files)
  - year=2025/month=3 (1 part-files)
  - year=2025/month=4 (1 part-files)
  - year=2025/month=5 (1 part-files)
  - year=2025/month=6 (1 part-files)
  - year=2025/month=7 (1 part-files)
  - year=2025/month=8 (1 part-files)
  - year=2025/month=9 (1 part-files)
+-------------+--------------+------------+
|service_level|july_shipments|july_fee_usd|
+-------------+--------------+------------+
|      Express|          8493|   322469.18|
|     Standard|          8494|   251849.49|
|      Economy|          8493|   214026.65|
+-------------+--------------+------------+

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- P

Trong mục (1) Scan parquet của kết quả explain(mode="formatted"), dòng PartitionFilters: [isnotnull(year#...), isnotnull(month#...), (year#... = 2025), (month#... = 7)] xuất hiện rõ ràng.   

Điều này chứng minh Spark áp dụng kỹ thuật Partition Pruning (cắt tỉa phân vùng): dựa vào cấu trúc thư mục year=2025/month=7, trình tối ưu Catalyst bỏ qua hoàn toàn 11 thư mục tháng còn lại trên đĩa mà không cần mở tệp hay giải nén dữ liệu bên trong.   

Câu 8. Thực nghiệm so sánh định dạng lưu trữ CSV và Parquet

In [13]:
COMPARE_COLS = [
    "shipment_id",
    "service_level",
    "destination_state",
    "year",
    "month",
    "weight_kg",
    "total_fee_usd",
]
subset = silver_clean.select(*COMPARE_COLS)

CSV_OUT = ROOT / "compare" / "csv"
PARQUET_OUT = ROOT / "compare" / "parquet"

# Xuất cùng tập cột và cùng số bản ghi với 4 part-file
(
    subset.coalesce(4)
    .write.mode("overwrite")
    .option("header", "true")
    .csv(str(CSV_OUT))
)
(
    subset.coalesce(4)
    .write.mode("overwrite")
    .parquet(str(PARQUET_OUT))
)

def folder_mib(folder: Path) -> float:
    total_b = sum(
        p.stat().st_size
        for p in folder.rglob("*")
        if p.is_file() and not p.name.startswith(".") and not p.name.startswith("_")
    )
    return round(total_b / (2**20), 2)

csv_mib = folder_mib(CSV_OUT)
parquet_mib = folder_mib(PARQUET_OUT)

COMPARE_SCHEMA = T.StructType([
    T.StructField("shipment_id", T.LongType()),
    T.StructField("service_level", T.StringType()),
    T.StructField("destination_state", T.StringType()),
    T.StructField("year", T.IntegerType()),
    T.StructField("month", T.IntegerType()),
    T.StructField("weight_kg", T.DoubleType()),
    T.StructField("total_fee_usd", T.DoubleType()),
])

def run_format_benchmark(fmt: str):
    start = time.perf_counter()
    if fmt == "CSV":
        df_fmt = (
            spark.read.schema(COMPARE_SCHEMA)
            .option("header", "true")
            .csv(str(CSV_OUT))
        )
    else:
        df_fmt = spark.read.parquet(str(PARQUET_OUT))

    res = (
        df_fmt.filter(F.col("month") == 7)
        .groupBy("service_level", "destination_state")
        .agg(F.sum("total_fee_usd").alias("total_fee"))
        .collect()
    )
    return time.perf_counter() - start, len(res)

# Chạy warm-up 1 lượt và đo lặp lại 3 lượt lấy trung vị
_ = run_format_benchmark("CSV")
_ = run_format_benchmark("Parquet")

csv_runs = [run_format_benchmark("CSV")[0] for _ in range(3)]
pq_runs = [run_format_benchmark("Parquet")[0] for _ in range(3)]

csv_med = round(statistics.median(csv_runs), 4)
pq_med = round(statistics.median(pq_runs), 4)

print("=== BẢNG SO SÁNH LƯU TRỮ VÀ TRUY VẤN: CSV VS PARQUET ===")
print(f"CSV     | Dung lượng: {csv_mib:6.2f} MiB | Trung vị: {csv_med:.4f} s | Các lượt chạy: {[round(x, 4) for x in csv_runs]}")
print(f"Parquet | Dung lượng: {parquet_mib:6.2f} MiB | Trung vị: {pq_med:.4f} s | Các lượt chạy: {[round(x, 4) for x in pq_runs]}")
print(f"Tỷ lệ nén dung lượng (CSV / Parquet): {csv_mib / parquet_mib:.2f}x")
print(f"Tỷ lệ tốc độ truy vấn (CSV / Parquet): {csv_med / pq_med:.2f}x")

=== BẢNG SO SÁNH LƯU TRỮ VÀ TRUY VẤN: CSV VS PARQUET ===
CSV     | Dung lượng:  11.09 MiB | Trung vị: 1.1859 s | Các lượt chạy: [1.1859, 1.2309, 0.7961]
Parquet | Dung lượng:   2.07 MiB | Trung vị: 0.3866 s | Các lượt chạy: [0.3866, 0.3829, 0.4348]
Tỷ lệ nén dung lượng (CSV / Parquet): 5.36x
Tỷ lệ tốc độ truy vấn (CSV / Parquet): 3.07x


Câu 9. Kiểm tra tính toàn vẹn dữ liệu và tính lũy đẳng (Idempotency)

In [14]:
# 1. Kiểm tra Idempotency: chạy lại bước ghi Curated với cùng đầu vào Cleaned
gold_rows_before = spark.read.parquet(str(GOLD_DIR)).count()

gold_rerun = (
    spark.read.parquet(str(SILVER_DIR))
    .groupBy("year", "month", "service_level", "destination_state")
    .agg(
        F.count("*").alias("shipments"),
        F.round(F.sum("total_fee_usd"), 2).alias("total_fee_usd"),
        F.round(F.avg("total_fee_usd"), 2).alias("avg_fee_usd"),
        F.sum(F.when(F.col("delivery_status") == "Late", 1).otherwise(0)).alias("late_shipments"),
        F.round(
            F.avg(F.when(F.col("delivery_status") == "Late", 1.0).otherwise(0.0)) * 100,
            2,
        ).alias("late_rate_pct"),
    )
)

(
    gold_rerun.write.mode("overwrite")
    .partitionBy("year", "month")
    .parquet(str(GOLD_DIR))
)

gold_after = spark.read.parquet(str(GOLD_DIR))
gold_rows_after = gold_after.count()

print(f"Số nhóm Curated trước khi chạy lại: {gold_rows_before:,}")
print(f"Số nhóm Curated sau khi chạy lại:   {gold_rows_after:,}")
assert gold_rows_before == gold_rows_after, "Vi phạm tính lũy đẳng: dữ liệu bị nhân đôi!"
print("-> Xác nhận Idempotency: Dữ liệu KHÔNG bị nhân đôi khi chạy lại pipeline với mode('overwrite').\n")

# 2. Đối chiếu số dòng Raw, Cleaned, Curated
raw_cnt = raw.count()
clean_df = spark.read.parquet(str(SILVER_DIR))
clean_cnt = clean_df.count()
curated_cnt = gold_rows_after

print(f"Đối chiếu số bản ghi: Raw = {raw_cnt:,} | Cleaned = {clean_cnt:,} | Curated = {curated_cnt:,} nhóm")

# 3. Đối chiếu tính nhất quán của 3 KPI nghiệp vụ giữa Cleaned và Curated
kpi_clean = clean_df.agg(
    F.count("*").alias("total_shipments"),
    F.round(F.sum("total_fee_usd"), 2).alias("total_fee"),
    F.sum(F.when(F.col("delivery_status") == "Late", 1).otherwise(0)).alias("total_late"),
).first()

kpi_gold = gold_after.agg(
    F.sum("shipments").alias("total_shipments"),
    F.round(F.sum("total_fee_usd"), 2).alias("total_fee"),
    F.sum("late_shipments").alias("total_late"),
).first()

print("\n=== ĐỐI CHIẾU KPI NGHIỆP VỤ (CLEANED VS CURATED) ===")
print(f"KPI 1 - Tổng số vận đơn:       Cleaned = {kpi_clean['total_shipments']:,} | Curated = {kpi_gold['total_shipments']:,}")
print(f"KPI 2 - Tổng chi phí (USD):    Cleaned = {kpi_clean['total_fee']:,.2f} | Curated = {kpi_gold['total_fee']:,.2f}")
print(f"KPI 3 - Tổng vận đơn giao trễ: Cleaned = {kpi_clean['total_late']:,} | Curated = {kpi_gold['total_late']:,}")

assert kpi_clean["total_shipments"] == kpi_gold["total_shipments"]
assert abs(kpi_clean["total_fee"] - kpi_gold["total_fee"]) < 1.0
assert kpi_clean["total_late"] == kpi_gold["total_late"]
print("\n[ĐẠT CHUẨN]: Toàn bộ KPI giữa tầng Cleaned và Curated nhất quán 100%!")

Số nhóm Curated trước khi chạy lại: 288
Số nhóm Curated sau khi chạy lại:   288
-> Xác nhận Idempotency: Dữ liệu KHÔNG bị nhân đôi khi chạy lại pipeline với mode('overwrite').

Đối chiếu số bản ghi: Raw = 300,000 | Cleaned = 300,000 | Curated = 288 nhóm

=== ĐỐI CHIẾU KPI NGHIỆP VỤ (CLEANED VS CURATED) ===
KPI 1 - Tổng số vận đơn:       Cleaned = 300,000 | Curated = 300,000
KPI 2 - Tổng chi phí (USD):    Cleaned = 9,272,678.70 | Curated = 9,272,678.70
KPI 3 - Tổng vận đơn giao trễ: Cleaned = 25,932 | Curated = 25,932

[ĐẠT CHUẨN]: Toàn bộ KPI giữa tầng Cleaned và Curated nhất quán 100%!


Câu 10. Tổng kết Pipeline thực nghiệm
1. Sơ đồ kiến trúc Pipeline ETL (Raw → Cleaned → Curated)

### Kiến trúc Pipeline Data Lake (Lab 02 – Bài 2)

```text
[Google Drive]
      │  (Copy/Export nguyên bản vào vùng lưu trữ cục bộ)
      ▼
┌─────────────────────────────────────────────────────────────────────────┐
│ TẦNG RAW (/content/lab02_bai2/data_lake/raw)                            │
│ • Tệp gốc: Lab02_Bai2_US_Logistics_Shipments.csv (300.000 dòng, 21 cột) │
│ • Đọc bằng PySpark với RAW_SCHEMA tường minh + parse scan_events_json   │
└───────────────────────────────┬─────────────────────────────────────────┘
                                │
         ┌──────────────────────┴──────────────────────┐
         ▼                                             ▼
┌──────────────────────────────┐        ┌─────────────────────────────────┐
│ NHÁNH BÁN CẤU TRÚC           │        │ DATA QUALITY GATE (6 TIÊU CHÍ)  │
│ • Lưu JSON Lines có cấu trúc │        │ • Khóa trùng, ngày, số đo, JSON │
│   (/json_events/part-*.json) │        │ • Công thức phí & trạng thái    │
│ • explode_outer(stages[])    │        │   -> Tách dòng lỗi (/quarantine)│
└──────────────────────────────┘        └──────────────┬──────────────────┘
                                                       │ (Bản ghi hợp lệ)
                                                       ▼
┌─────────────────────────────────────────────────────────────────────────┐
│ TẦNG CLEANED / SILVER (/content/lab02_bai2/data_lake/cleaned)           │
│ • Chuẩn hóa DateType, loại trùng shipment_id, bỏ 2 cột gây nhiễu        │
│ • Bổ sung year, month -> Ghi Parquet & Partitioned Parquet (year/month) │
└──────────────────────────────────────────────────────┬──────────────────┘
                                                       │ (GroupBy KPI)
                                                       ▼
┌─────────────────────────────────────────────────────────────────────────┐
│ TẦNG CURATED / GOLD (/content/lab02_bai2/data_lake/curated)             │
│ • Bảng tổng hợp theo (year, month, service_level, destination_state)    │
│ • KPI: Số vận đơn, Tổng phí USD, Phí bình quân, Tỷ lệ giao trễ (%)     │
│ • Ghi Parquet phân vùng year/month + Đối soát KPI & Idempotency         │
└─────────────────────────────────────────────────────────────────────────┘
```

JSON Lines (json_events): Lưu trữ mỗi bản ghi thành một đối tượng JSON độc lập trên từng dòng, giữ nguyên cấu trúc phân cấp lồng nhau (scan.hub, scan.mode, mảng stages[]) mà không cần làm phẳng cứng nhắc, phù hợp làm định dạng trao đổi sự kiện bán cấu trúc và ghi song song nhiều tệp part-*.   

Parquet (Cleaned và Curated): Lưu trữ dữ liệu hướng cột (columnar format) kèm nén tự động và siêu dữ liệu thống kê. Khi kết hợp phân vùng thư mục partitionBy("year", "month"), Parquet giúp giảm mạnh dung lượng đĩa và tăng tốc truy vấn phân tích nhờ hai cơ chế Partition Pruning (bỏ qua các thư mục tháng không truy vấn) và Column Pruning (chỉ đọc các cột tham gia phép tính).  